# Recorte de gradiente (gradient clipping)

(Ejercicio, 2 puntos posibles)

El recorte de gradiente limita el tamaño de $\nabla f$ **antes** de actualizar los pesos $w$. En un valle elíptico el eje $w_2$ es muy empinado: un $\eta$ grande hace que el descenso por gradiente oscile o se dispare. Si recortas la norma del gradiente, puedes usar el mismo $\eta$ sin explotar.

En este ejercicio hay dos tareas:

1. **(1 punto)** Implementar el descenso por gradiente con recorte de la norma.
2. **(1 punto)** Configurar `learning_rate`, `clip_norm` y `max_iterations` para acercarte al óptimo $(0, 0)$ por debajo del umbral. Puedes usar hasta 2000 iteraciones.

El valle elíptico, la visualización y el ciclo `optimize` ya están dados.

INSTRUCCIONES: Completa el código faltante.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irvingvasquez/cv2course_intro_nn/blob/master/23_opt_grad_clipping.ipynb)

@juan1rving


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.animation as animation
import nni


In [ ]:
# --- Criterio de evaluación (fijo: no lo modifiques) ---
# Pesos iniciales
w_inicial = [-4.0, 0.4]
# Distancia euclidiana máxima al óptimo (0, 0)
umbral = 0.1
# Tope de iteraciones para la evaluación
max_iteraciones_permitidas = 2000


In [ ]:
# TODO (1 punto): configura learning_rate, clip_norm y max_iterations
# para acercarte al óptimo. Se considera que llegaste si, en no más
# de 2000 pasos, la distancia a (0, 0) es menor que umbral.
# GD y el recorte usarán el mismo learning_rate.
learning_rate = 0.01
clip_norm = 10.0
max_iterations = 100


## Problema

Usaremos un **valle elíptico**: una cuadrática con curvaturas muy distintas. Un eje es casi plano y el otro es muy empinado. El gradiente en $w_2$ es grande; sin recorte, un $\eta$ útil para $w_1$ hace que $w_2$ se dispare.

$$
f(w_1, w_2) = \tfrac{1}{2}\bigl(a w_1^{2} + b w_2^{2}\bigr)
$$

$$
\frac{\partial f}{\partial w_1} = a w_1, \qquad \frac{\partial f}{\partial w_2} = b w_2
$$

con $a = 1$ y $b = 40$. El mínimo global está en $w = (0, 0)$ y $f(w)=0$.

El descenso por gradiente clásico zigzaguea o explota en $w_2$. El recorte acota $\|\nabla f\|$ y deja un paso estable en ambas coordenadas.


In [ ]:
problem = nni.EllipticValleyProblem()

def square_view(xs, ys, margin=0.20):
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    dx = x_max - x_min
    dy = y_max - y_min
    span = max(dx, dy, 1e-6)
    pad_x = margin * (dx if dx > 0 else span)
    pad_y = margin * (dy if dy > 0 else span)
    x_min, x_max = x_min - pad_x, x_max + pad_x
    y_min, y_max = y_min - pad_y, y_max + pad_y
    side = max(x_max - x_min, y_max - y_min)
    cx = 0.5 * (x_min + x_max)
    cy = 0.5 * (y_min + y_max)
    half = side / 2.0
    return cx - half, cx + half, cy - half, cy + half

min_w1, max_w1, min_w2, max_w2 = square_view(
    [w_inicial[0], problem.optimal_params[0]],
    [w_inicial[1], problem.optimal_params[1]],
)

w1 = np.linspace(min_w1, max_w1, 200)
w2 = np.linspace(min_w2, max_w2, 200)
W1, W2 = np.meshgrid(w1, w2)
Z = problem.function(W1, W2)

max_iterations_run = min(int(max_iterations), max_iteraciones_permitidas)
gd = nni.GradientDescentOptimizer(
    learning_rate=learning_rate,
    max_iterations=max_iterations_run,
    tolerance=1e-6,
)
gd.optimize(
    problem.function,
    lambda w1, w2: [problem.gradient_x(w1, w2), problem.gradient_y(w1, w2)],
    w_inicial,
)
gd_w1 = [log['params'][0] for log in gd.get_history()]
gd_w2 = [log['params'][1] for log in gd.get_history()]
gd_z = [log['value'] for log in gd.get_history()]


In [ ]:
# Create a figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

ax1.contourf(W1, W2, Z, levels=50, cmap='viridis')
ax1.plot(w_inicial[0], w_inicial[1], 'wo', markeredgecolor='k')
ax1.plot(problem.optimal_params[0], problem.optimal_params[1], 'ro')
ax1.plot(gd_w1, gd_w2, 'C1.-', markersize=5, label=f'GD, lr={gd.learning_rate}, iter={len(gd_w1)}')
ax1.set_xlim(min_w1, max_w1)
ax1.set_ylim(min_w2, max_w2)
ax1.set_aspect('equal', adjustable='box')
ax1.set_xlabel(r'$w_1$')
ax1.set_ylabel(r'$w_2$')
ax1.set_title('Contour Plot')
ax1.legend(loc='upper right')

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(W1, W2, Z, cmap='viridis', alpha=0.7)
ax2.plot(gd_w1, gd_w2, gd_z, 'C1.-', markersize=5, label='GD')
ax2.scatter(w_inicial[0], w_inicial[1], problem.function(*w_inicial),
            c='w', s=40, edgecolors='k')
ax2.scatter(problem.optimal_params[0], problem.optimal_params[1], 0, c='r', s=40)
ax2.set_xlim(min_w1, max_w1)
ax2.set_ylim(min_w2, max_w2)
ax2.set_xlabel(r'$w_1$')
ax2.set_ylabel(r'$w_2$')
ax2.set_zlabel('z')
ax2.set_title('3D Projection')

plt.tight_layout()
plt.show()


## Actualización con recorte de norma

El descenso por gradiente clásico usa el gradiente crudo:

$$
w_{t} = w_{t-1} - \eta \nabla f(w_{t-1})
$$

Con recorte de norma se acota $\|g_t\|$ a un tope $c$ (`clip_norm`) y luego se da el paso:

$$
\tilde{g}_{t} =
\begin{cases}
g_{t} & \text{si } \|g_{t}\| \le c \\
c\,\dfrac{g_{t}}{\|g_{t}\|} & \text{si } \|g_{t}\| > c
\end{cases}
$$

$$
w_{t} = w_{t-1} - \eta \tilde{g}_{t}
$$

La clase `nni.GradientDescentOptimizer` ya implementa el ciclo `optimize`. Tú debes completar `GradientClippingOptimizer`: guardar `clip_norm` y sobrescribir `next_params`.


In [ ]:
class GradientClippingOptimizer(nni.GradientDescentOptimizer):
    def __init__(self, learning_rate=0.01, max_iterations=1000, tolerance=1e-6, clip_norm=10.0):
        super().__init__(learning_rate=learning_rate, max_iterations=max_iterations, tolerance=tolerance)
        self.name = 'Gradient clipping'
        # TODO (1 punto): implementa el recorte de la norma del gradiente.
        # Guarda el tope c (clip_norm).
        self.clip_norm = None

    def next_params(self, gradient, w):
        # Completa la actualización:
        # 1) Si ||gradiente|| > clip_norm, recórtalo a norma clip_norm.
        # 2) Da el paso restando learning_rate * gradiente recortado.
        return None


In [ ]:
max_iterations_run = min(int(max_iterations), max_iteraciones_permitidas)
if max_iterations > max_iteraciones_permitidas:
    print(f'Aviso: max_iterations se recortó a {max_iteraciones_permitidas}.')

optimizer = GradientClippingOptimizer(
    learning_rate=learning_rate,
    max_iterations=max_iterations_run,
    tolerance=1e-6,
    clip_norm=clip_norm,
)

w = optimizer.optimize(
    problem.function,
    lambda w1, w2: [problem.gradient_x(w1, w2), problem.gradient_y(w1, w2)],
    w_inicial,
)

history = optimizer.get_history()
k = min(len(history), max_iteraciones_permitidas)
w_eval = np.array(history[k - 1]['params'], dtype=float)
w_optimo = np.array(problem.optimal_params, dtype=float)
distancia = np.linalg.norm(w_eval - w_optimo)
llego = np.isfinite(distancia) and distancia < umbral

print(f'w evaluado: {w_eval}')
print(f'w óptimo: {w_optimo}')
print(f'Mejor valor encontrado: {problem.function(*w_eval)}')
print(f'Distancia al óptimo: {distancia:.4f}  (umbral = {umbral})')
print(f'Iteraciones: {k}  (máximo permitido = {max_iteraciones_permitidas})')
if llego:
    print('Evaluación: SÍ se acerca al óptimo.')
else:
    print('Evaluación: NO se acerca al óptimo. Ajusta learning_rate, clip_norm y max_iterations.')


In [ ]:
# Comparación visual: mismo learning_rate e iteraciones
gd = nni.GradientDescentOptimizer(
    learning_rate=learning_rate,
    max_iterations=max_iterations_run,
    tolerance=1e-6,
)
gd.optimize(
    problem.function,
    lambda w1, w2: [problem.gradient_x(w1, w2), problem.gradient_y(w1, w2)],
    w_inicial,
)

def path_w(opt):
    h = opt.get_history()
    return [log['params'][0] for log in h], [log['params'][1] for log in h]

gd_w1, gd_w2 = path_w(gd)
clip_w1, clip_w2 = path_w(optimizer)

fig, ax = plt.subplots(figsize=(6, 6))
ax.contourf(W1, W2, Z, levels=50, cmap='viridis')
ax.plot(w_inicial[0], w_inicial[1], 'wo', markeredgecolor='k')
ax.plot(problem.optimal_params[0], problem.optimal_params[1], 'ro')
ax.plot(gd_w1, gd_w2, 'C1.-', markersize=5, label=f'GD, lr={gd.learning_rate}, iter={len(gd_w1)}')
ax.plot(clip_w1, clip_w2, 'w.-', markersize=5, label=f'{optimizer.name}, lr={optimizer.learning_rate}, c={optimizer.clip_norm}, iter={len(clip_w1)}')
ax.set_xlim(min_w1, max_w1)
ax.set_ylim(min_w2, max_w2)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel(r'$w_1$')
ax.set_ylabel(r'$w_2$')
ax.set_title('GD vs recorte (mismo lr)')
ax.legend(loc='upper right')
plt.show()


In [ ]:
# Animate and save the clipped path
fig, ax = plt.subplots(figsize=(6, 6))
ax.contourf(W1, W2, Z, levels=50, cmap='viridis')
ax.plot(clip_w1[0], clip_w2[0], 'wo', markeredgecolor='k')
ax.plot(problem.optimal_params[0], problem.optimal_params[1], 'ro')
(line,) = ax.plot([], [], 'w.-', markersize=8)
ax.set_xlim(min_w1, max_w1)
ax.set_ylim(min_w2, max_w2)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel(r'$w_1$')
ax.set_ylabel(r'$w_2$')
title = ax.set_title('')

def update(frame):
    line.set_data(clip_w1[:frame + 1], clip_w2[:frame + 1])
    title.set_text(f'{optimizer.name} lr = {optimizer.learning_rate}, c = {optimizer.clip_norm}, iter {frame}/{len(clip_w1)}')
    return (line, title)

animation.FuncAnimation(fig, update, frames=len(clip_w1), interval=30, blit=True, repeat=False).save('grad_clipping.mp4')
